# Drama -- Google Colab Training

Train Drama on a Colab A100 from scratch. All dependencies and compatibility patches are handled automatically.

**Before running:** `Runtime -> Change runtime type -> A100 GPU` (requires Colab Pro)

Run cells 1-8 in order. Once training starts (cell 8), open a second browser tab connected to the same runtime and run cell 9 in parallel to back up checkpoints to Google Drive.

In [ ]:
# Step 1 -- Verify GPU
!nvidia-smi

In [ ]:
# Step 2 -- Mount Google Drive (checkpoints will be backed up here)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/drama_checkpoints'
!mkdir -p {DRIVE_CKPT}
print(f'Checkpoint backup directory: {DRIVE_CKPT}')

In [ ]:
# Step 3 -- Clone repository (always starts fresh to avoid stale patches)
import os, shutil
if os.path.exists('/content/Drama'):
    shutil.rmtree('/content/Drama')
!git clone https://github.com/realwenlongwang/Drama /content/Drama

# Reorganise into src/ to match the repository layout
src = '/content/Drama/src'
os.makedirs(src, exist_ok=True)
for name in ['agents.py', 'train.py', 'eval.py', 'env_wrapper.py',
             'replay_buffer.py', 'tools.py', 'utils.py',
             'atari_performance.csv', 'config_files', 'envs',
             'sub_models', 'mamba_ssm', 'csrc']:
    shutil.move(f'/content/Drama/{name}', f'{src}/{name}')
print('Repository organised into src/')

In [ ]:
# Step 4 -- System dependencies
!apt-get install -y ffmpeg libsm6 libxext6 -q

In [ ]:
# Step 5 -- Verify PyTorch (Colab pre-installs it)
import torch
print(f'Torch:  {torch.__version__}')
print(f'CUDA:   {torch.cuda.is_available()}')
print(f'GPU:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Step 6 -- Install Python dependencies
# causal-conv1d and mamba-ssm compile CUDA extensions that require torch at build time;
# --no-build-isolation lets them see the already-installed torch instead of an empty sandbox.
!pip install -q packaging
!pip install causal-conv1d --no-build-isolation -q
!pip install mamba-ssm --no-build-isolation -q
!pip install -q \
    yacs==0.1.8 \
    tensorboardX==2.6.2.2 \
    moviepy==1.0.3 \
    colorama==0.4.6 \
    einops==0.8.0 \
    tqdm \
    "opencv-python-headless>=4.10.0" \
    "gymnasium[atari]" \
    kornia==0.7.3 \
    wandb \
    pytorch_warmup==0.1.1 \
    pandas==2.2.2 \
    line_profiler
!pip install -q "autorom[accept-rom-license]"
!AutoROM --accept-license --quiet
print('All packages installed.')

In [ ]:
# Step 7 -- Apply compatibility patches
# Drama was written for causal-conv1d 1.2.x and gymnasium 0.29.x.
# Colab ships newer versions whose APIs differ; the patches below bridge the gaps.
# All patches are idempotent -- safe to re-run.

# --- 1. envs/my_atari.py: gymnasium >= 1.0 removed automatic ALE env registration ---
path = '/content/Drama/src/envs/my_atari.py'
with open(path) as f:
    txt = f.read()
if 'register_envs' not in txt:
    txt = txt.replace(
        'import gymnasium as gym',
        'import gymnasium as gym\nimport ale_py\ngym.register_envs(ale_py)',
        1
    )
    with open(path, 'w') as f:
        f.write(txt)
print('my_atari.py              patched')

# --- 2. eval.py: gymnasium >= 1.0 requires ResizeObservation to receive a tuple shape ---
path = '/content/Drama/src/eval.py'
with open(path) as f:
    txt = f.read()
if 'shape=(image_size, image_size)' not in txt:
    txt = txt.replace('shape=image_size', 'shape=(image_size, image_size)')
    with open(path, 'w') as f:
        f.write(txt)
print('eval.py                  patched')

# --- 3. mamba2.py: disable fused triton path, use causal_conv1d_fn wrapper instead ---
# causal-conv1d >= 1.4 changed the C API used by the fused path in ssd_combined.py.
# Setting use_mem_eff_path=False routes all calls through the Python wrapper
# (causal_conv1d_fn) which handles version differences internally.
path = '/content/Drama/src/mamba_ssm/modules/mamba2.py'
with open(path) as f:
    txt = f.read()
if 'self.use_mem_eff_path = False  # patched' not in txt:
    txt = txt.replace(
        'self.use_mem_eff_path = use_mem_eff_path',
        'self.use_mem_eff_path = False  # patched: causal-conv1d >= 1.4 C API incompatibility'
    )
    with open(path, 'w') as f:
        f.write(txt)
print('mamba2.py                patched')

# --- 4. selective_scan_interface.py: causal_conv1d_fwd gained has_initial_states in >= 1.4 ---
path = '/content/Drama/src/mamba_ssm/ops/selective_scan_interface.py'
with open(path) as f:
    txt = f.read()
if '    x, conv1d_weight, conv1d_bias, None, None, None, None, True' not in txt:
    txt = txt.replace(
        '    x, conv1d_weight, conv1d_bias, None, None, None, True',
        '    x, conv1d_weight, conv1d_bias, None, None, None, None, True'
    )
    with open(path, 'w') as f:
        f.write(txt)
print('selective_scan_interface.py patched')

print('\nAll patches applied.')

In [ ]:
# Step 8 -- Start training
# Trains from scratch on ALE/Pong-v5 for 100k steps (~105k total to ensure the last
# episode finishes). Checkpoints are saved every 2000 steps under src/saved_models/.
# WandB runs in offline mode; no API key is required.
import os
os.environ['WANDB_MODE'] = 'offline'
%cd /content/Drama/src

!python train.py --Wandb.Init.Mode offline

In [ ]:
# Step 9 -- Auto-backup checkpoints to Drive (run in a separate tab while training)
# Open a second Colab tab, connect to the same runtime, and run only this cell.
# It wakes every 10 minutes and copies any .pth files to Google Drive.
import glob, os, shutil, time

DRIVE_CKPT = '/content/drive/MyDrive/drama_checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

for cycle in range(200):
    time.sleep(600)
    synced = 0
    for src in glob.glob('/content/Drama/src/saved_models/**/*.pth', recursive=True):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(src))
        shutil.copy2(src, dst)
        synced += 1
    print(f'[Backup {cycle + 1}] {synced} file(s) synced to Drive')